# 10 · JobRole ↔ O*NET Manual Mapping

**Project:** Enterprise HR AI  
**Task:** Create `data/external/jobrole_onet_mapping.csv` — a manually curated lookup table  
mapping the 9 IBM JobRole categories to real O*NET-SOC codes from `occupation_master.csv`.

> **No fuzzy-matching or algorithmic similarity scoring is used here.**  
> Step 4 established that 4 of 9 IBM roles have zero legitimate O*NET counterpart.  
> All codes were verified against actual `occupation_master.csv` rows before use.  
> No O*NET-SOC code is invented or extrapolated.

---

In [1]:
import pandas as pd
import os

PROC = os.path.join('..', 'data', 'processed')
EXT  = os.path.join('..', 'data', 'external')
os.makedirs(EXT, exist_ok=True)

# Load occupation_master to verify every SOC code we use actually exists
occ = pd.read_csv(os.path.join(PROC, 'occupation_master.csv'))
soc_col   = [c for c in occ.columns if 'soc' in c.lower() or 'code' in c.lower()][0]
title_col = [c for c in occ.columns if 'title' in c.lower()][0]
valid_soc_codes = set(occ[soc_col].str.strip())
print(f'occupation_master: {len(occ):,} rows, {len(valid_soc_codes):,} unique SOC codes')
print(f'SOC col: {soc_col!r}  |  Title col: {title_col!r}')

occupation_master: 1,016 rows, 1,016 unique SOC codes
SOC col: 'O*NET-SOC Code'  |  Title col: 'Title'


---
## Mapping Rationale Per Role

Each mapping was verified against `occupation_master.csv` before inclusion.
Confidence levels: `medium` (plausible domain match), `low` (approximate, cross-domain),
`very_low` (structurally unmappable — IBM role is too generic).
No `high` confidence rows exist — this was established in Step 4.

| IBM Role | Decision |
|:---|:---|
| Healthcare Representative | No direct O*NET role; nearest is `41-3091.00` Services Sales Rep — same function (relationship-based selling), healthcare sector |
| Human Resources | Two valid O*NET candidates: `13-1071.00` Specialist vs `11-3121.00` Manager. IBM role title is non-specific; using Specialist as default |
| Laboratory Technician | `29-2012.00` Medical and Clinical Laboratory Technicians — closest title-level match; other lab tech codes (19-4031, 19-4021) are chemistry/biology-specific |
| Manager | Structurally unmappable — IBM 'Manager' spans 8+ O*NET manager codes; mapping to any single one would be false confidence |
| Manufacturing Director | No 'Manufacturing Director' in O*NET; `11-3051.00` Industrial Production Managers is the closest production-level leadership role |
| Research Director | No 'Research Director' in O*NET; `11-9121.00` Natural Sciences Managers is the closest R&D leadership role |
| Research Scientist | `15-1221.00` Computer and Information Research Scientists — confirmed in Step 4 as a near-miss; valid when EducationField is technical/CS |
| Sales Executive | `11-2022.00` Sales Managers — closest leadership-level sales role; 'Executive' implies seniority/quota ownership rather than individual contributor |
| Sales Representative | `41-3091.00` Sales Representatives of Services — one of the 4 Step 4 near-misses; broadest non-technical services match for IBM's B2B/HR tech context |


In [2]:
# -------------------------------------------------------------------
# Manually curated mapping — every SOC code verified against
# occupation_master.csv before inclusion. Confidence is deliberately
# capped at 'medium'; no 'high' exists per Step 4 analysis.
# -------------------------------------------------------------------

rows = [
    {
        'ibm_job_role': 'Healthcare Representative',
        'onet_soc_code': '41-3091.00',
        'onet_title': 'Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel',
        'match_confidence': 'low',
        'mapping_note': ('No direct O*NET equivalent for healthcare-sector liaison/sales roles; '
                         '"Sales Representatives of Services" (41-3091.00) is the broadest service-sector '
                         'sales code in occupation_master.csv and captures the relationship-selling '
                         'function, but does not reflect the clinical or regulatory context of a '
                         'pharmaceutical/device healthcare representative.')
    },
    {
        'ibm_job_role': 'Human Resources',
        'onet_soc_code': '13-1071.00',
        'onet_title': 'Human Resources Specialists',
        'match_confidence': 'medium',
        'mapping_note': ('"Human Resources Specialists" (13-1071.00) is the best-fit individual-contributor '
                         'HR role in occupation_master.csv; "Human Resources Managers" (11-3121.00) also '
                         'exists but implies supervisory authority not implied by the IBM job role label alone; '
                         'Specialist is used as the conservative default — role-level can be refined using '
                         'JobLevel or YearsAtCompany if needed.')
    },
    {
        'ibm_job_role': 'Laboratory Technician',
        'onet_soc_code': '29-2012.00',
        'onet_title': 'Medical and Clinical Laboratory Technicians',
        'match_confidence': 'medium',
        'mapping_note': ('"Medical and Clinical Laboratory Technicians" (29-2012.00) is the closest title-level '
                         'match in occupation_master.csv; chemistry (19-4031.00) and biological (19-4021.00) '
                         'technician codes also exist but presuppose a specific scientific discipline not '
                         'determinable from the IBM label alone; 29-2012.00 is chosen as the broadest '
                         'lab-bench technician match across pharma/biotech/clinical contexts.')
    },
    {
        'ibm_job_role': 'Manager',
        'onet_soc_code': '11-9199.00',
        'onet_title': 'Managers, All Other',
        'match_confidence': 'very_low',
        'mapping_note': ('IBM "Manager" is too generic to map meaningfully: occupation_master.csv contains '
                         '52+ O*NET manager codes spanning sales, HR, IT, production, logistics, and more; '
                         'the catch-all code "Managers, All Other" (11-9199.00) is used solely as a '
                         'placeholder — role-intelligence features for this group must rely on the '
                         'Department field in the employee record rather than O*NET data, '
                         'per the Step 4 architectural decision.')
    },
    {
        'ibm_job_role': 'Manufacturing Director',
        'onet_soc_code': '11-3051.00',
        'onet_title': 'Industrial Production Managers',
        'match_confidence': 'low',
        'mapping_note': ('No "Manufacturing Director" title exists in occupation_master.csv; '
                         '"Industrial Production Managers" (11-3051.00) is the closest production-leadership '
                         'role covering planning, directing, and coordinating manufacturing activities; '
                         '"General and Operations Managers" (11-1021.00) was considered but is broader '
                         'and less domain-specific than a plant-level manufacturing director role.')
    },
    {
        'ibm_job_role': 'Research Director',
        'onet_soc_code': '11-9121.00',
        'onet_title': 'Natural Sciences Managers',
        'match_confidence': 'low',
        'mapping_note': ('No "Research Director" title exists in occupation_master.csv; '
                         '"Natural Sciences Managers" (11-9121.00) is the only R&D-leadership code '
                         'in occupation_master.csv that covers planning, directing, and coordinating '
                         'research activities; "Computer and Information Systems Managers" (11-3021.00) '
                         'was considered but is technology-operations focused rather than research-leadership.')
    },
    {
        'ibm_job_role': 'Research Scientist',
        'onet_soc_code': '15-1221.00',
        'onet_title': 'Computer and Information Research Scientists',
        'match_confidence': 'medium',
        'mapping_note': ('"Computer and Information Research Scientists" (15-1221.00) was identified as '
                         'a near-miss in Step 4 and is the strongest match when EducationField indicates '
                         'a technical/computational background; note this mapping carries ambiguity when '
                         'EducationField is Life Sciences or Medical — in those cases "Medical Scientists, '
                         'Except Epidemiologists" (19-1042.00) from occupation_master.csv would be a '
                         'better analog, but a single mapping row cannot capture both branches.')
    },
    {
        'ibm_job_role': 'Sales Executive',
        'onet_soc_code': '11-2022.00',
        'onet_title': 'Sales Managers',
        'match_confidence': 'low',
        'mapping_note': ('No "Sales Executive" title exists in occupation_master.csv; '
                         '"Sales Managers" (11-2022.00) is chosen because the "Executive" qualifier '
                         'implies quota-ownership, account strategy, and cross-functional leadership '
                         'rather than individual territory selling, making it structurally closer to '
                         'a management role than to "Sales Representatives of Services" (41-3091.00) '
                         'used for the Sales Representative row.')
    },
    {
        'ibm_job_role': 'Sales Representative',
        'onet_soc_code': '41-3091.00',
        'onet_title': 'Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel',
        'match_confidence': 'medium',
        'mapping_note': ('"Sales Representatives of Services" (41-3091.00) was one of the 4 Step 4 '
                         'near-miss sales rep codes in occupation_master.csv; chosen over '
                         '"Sales Representatives, Wholesale and Manufacturing" (41-4012.00) because '
                         'IBM\'s HR-tech/pharma dataset context implies service-sector B2B selling '
                         'rather than wholesale product distribution.')
    },
]

mapping_df = pd.DataFrame(rows)
print(mapping_df.to_string(index=False))
print()
print(f'Total rows: {len(mapping_df)}')

             ibm_job_role onet_soc_code                                                                                       onet_title match_confidence                                                                                                                                                                                                                                                                                                                                                                                                                                                                  mapping_note
Healthcare Representative    41-3091.00 Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel              low                                                                                                                       No direct O*NET equivalent for healthcare-sector liaison/sales roles; "Sales Representatives of Services" (41-30

In [3]:
print('=== VALIDATING ALL SOC CODES AGAINST occupation_master.csv ===')
all_ok = True
for _, row in mapping_df.iterrows():
    code_val = row['onet_soc_code']
    exists = code_val in valid_soc_codes
    status = 'OK' if exists else 'NOT FOUND IN occupation_master.csv -- INVALID'
    print(f'  [{status}] {code_val}  |  {row["onet_title"][:60]}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('CONFIRMED: All 9 SOC codes verified as present in occupation_master.csv. No invented codes.')
else:
    print('ERROR: One or more SOC codes not found in occupation_master.csv. Fix before saving.')
    raise AssertionError('Invalid SOC code detected')

=== VALIDATING ALL SOC CODES AGAINST occupation_master.csv ===
  [OK] 41-3091.00  |  Sales Representatives of Services, Except Advertising, Insur
  [OK] 13-1071.00  |  Human Resources Specialists
  [OK] 29-2012.00  |  Medical and Clinical Laboratory Technicians
  [OK] 11-9199.00  |  Managers, All Other
  [OK] 11-3051.00  |  Industrial Production Managers
  [OK] 11-9121.00  |  Natural Sciences Managers
  [OK] 15-1221.00  |  Computer and Information Research Scientists
  [OK] 11-2022.00  |  Sales Managers
  [OK] 41-3091.00  |  Sales Representatives of Services, Except Advertising, Insur

CONFIRMED: All 9 SOC codes verified as present in occupation_master.csv. No invented codes.


In [4]:
out_path = os.path.join(EXT, 'jobrole_onet_mapping.csv')
mapping_df.to_csv(out_path, index=False, encoding='utf-8')
print(f'Saved: {os.path.abspath(out_path)}')
print(f'Size : {os.path.getsize(out_path):,} bytes')
print()
print('=== FULL CSV CONTENTS ===')
roundtrip = pd.read_csv(out_path)
print(roundtrip.to_string(index=False))

Saved: C:\Users\ASUS\Desktop\enterprise_hr_ai\data\external\jobrole_onet_mapping.csv
Size : 4,325 bytes

=== FULL CSV CONTENTS ===
             ibm_job_role onet_soc_code                                                                                       onet_title match_confidence                                                                                                                                                                                                                                                                                                                                                                                                                                                                  mapping_note
Healthcare Representative    41-3091.00 Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel              low                                                                                                    

In [5]:
print('=== CONFIDENCE LEVEL SUMMARY ===')
conf_counts = mapping_df['match_confidence'].value_counts()
for level, count in conf_counts.items():
    print(f'  {level:<12}: {count} row(s)')

print()
# Enforce: zero 'high' confidence rows
if 'high' in conf_counts.index:
    n_high = conf_counts['high']
    print(f'PROBLEM: {n_high} row(s) claim confidence="high" — this is invalid.')
    print('Step 4 established that none of the 9 IBM roles have a direct O*NET counterpart.')
    print('Review and downgrade these rows before using in production.')
else:
    print('CONFIRMED: Zero rows claim confidence="high" -- consistent with Step 4 finding.')

print()
print('=== ROLE USABILITY GUIDE FOR DAY 3 ===')
usability = {
    'medium': 'Use O*NET skill data with a note that role mapping is approximate.',
    'low': 'Use O*NET skill data only as a broad reference; caveat results prominently.',
    'very_low': 'Do NOT use O*NET data for this role. Use Department field for skill recommendations instead.',
}
for _, row in mapping_df.iterrows():
    guide = usability[row['match_confidence']]
    print(f'  {row["ibm_job_role"]:<30} [{row["match_confidence"]}]  -> {guide}')

=== CONFIDENCE LEVEL SUMMARY ===
  low         : 4 row(s)
  medium      : 4 row(s)
  very_low    : 1 row(s)

CONFIRMED: Zero rows claim confidence="high" -- consistent with Step 4 finding.

=== ROLE USABILITY GUIDE FOR DAY 3 ===
  Healthcare Representative      [low]  -> Use O*NET skill data only as a broad reference; caveat results prominently.
  Human Resources                [medium]  -> Use O*NET skill data with a note that role mapping is approximate.
  Laboratory Technician          [medium]  -> Use O*NET skill data with a note that role mapping is approximate.
  Manager                        [very_low]  -> Do NOT use O*NET data for this role. Use Department field for skill recommendations instead.
  Manufacturing Director         [low]  -> Use O*NET skill data only as a broad reference; caveat results prominently.
  Research Director              [low]  -> Use O*NET skill data only as a broad reference; caveat results prominently.
  Research Scientist             [medium]  -> U